# Continuous-time range-only localization with WNOA

This example estimates a 3D trajectory from range measurements to known landmarks. The trajectory is represented by positions and velocities at four knot times. A white-noise-on-acceleration (WNOA) prior connects consecutive knots, while interpolation factors incorporate ranges measured between them.

The positions use `AugmentedPoint3` and the measurements use `KnownLandmarkSquaredRangeFactorAugmentedPoint3`, both from `gtsam_unstable`. We solve the resulting nonlinear least-squares problem with GTSAM's Levenberg–Marquardt optimizer.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All Rights Reserved. Authors: Frank Dellaert, et al. (see THANKS for the full author list). See LICENSE for the license information.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam_unstable/slam/doc/RangeOnlyWnoaAugmentedPoint.ipynb)

In [ ]:
try:
    import google.colab
except ImportError:
    pass

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import plotly.graph_objects as go
import gtsam
import gtsam_unstable as gu

## Range measurements and trajectory

We simulate a smooth trajectory observed by four fixed landmarks. Range measurements are taken at the first knot and at two times inside each subsequent knot interval. The knot positions and velocities are the only unknowns; landmark locations remain fixed.

For a landmark at $L_i$ and measured distance $d_i$, the squared-range factor uses

$$
e_i(p(t)) = \|p(t)-L_i\|^2-d_i^2.
$$

If the raw range has standard deviation $\sigma_r$, first-order uncertainty propagation gives $\sigma_{d^2}\approx 2d_i\sigma_r$. We use this value for the one-dimensional factor noise model, since the residual is expressed in squared meters.

In [4]:
rng = np.random.default_rng(7)

knot_times = np.array([0.0, 1.0, 2.0, 3.0])
measurement_times = np.array([0.0, 0.25, 0.75, 1.25, 1.75, 2.25, 2.75])
landmarks = np.array([
    [0.0, 0.0, 0.0],
    [3.0, 0.0, 0.0],
    [0.0, 3.0, 0.0],
    [0.0, 0.0, 3.0],
])
range_sigma = 0.02  # m
q_psd = np.full(3, 0.2)  # WNOA acceleration power spectral density


def true_position(t):
    t = np.asarray(t)
    return np.stack((
        0.6 + 0.35 * t + 0.06 * np.sin(0.9 * t),
        -0.2 + 0.1 * t - 0.07 * (1 - np.cos(0.7 * t)),
        0.7 - 0.15 * t + 0.06 * np.sin(1.1 * t),
    ), axis=-1)


def true_velocity(t):
    t = np.asarray(t)
    return np.stack((
        0.35 + 0.054 * np.cos(0.9 * t),
        0.1 - 0.049 * np.sin(0.7 * t),
        -0.15 + 0.066 * np.cos(1.1 * t),
    ), axis=-1)


distances = np.linalg.norm(
    true_position(measurement_times)[:, None, :] - landmarks[None, :, :],
    axis=2,
)
measured_ranges = distances + rng.normal(0, range_sigma, distances.shape)

## Build the factor graph

Each knot has a position key and a velocity key. `WnoaMotionFactorAugmentedPoint3` constrains consecutive knots using the WNOA prior.

A range at the first knot is connected directly to its position. For each intermediate measurement, `WnoaInterpFactorAugmentedPoint3` wraps the squared-range factor and propagates its constraint to the surrounding knot positions and velocities. There are no optimization variables at the intermediate measurement times.

In [5]:
states = [
    gtsam.StateData(gtsam.symbol("p", i), gtsam.symbol("v", i), float(t))
    for i, t in enumerate(knot_times)
]

initial = gtsam.Values()
for state, t in zip(states, knot_times):
    position = true_position(t) + np.array([0.15, -0.10, 0.13])
    velocity = true_velocity(t) + np.array([-0.10, 0.08, -0.06])
    gu.insertAugmentedPoint3(state.pose, gu.AugmentedPoint3(position), initial)
    initial.insert(state.velocity, velocity)

graph = gtsam.NonlinearFactorGraph()
for left, right in zip(states[:-1], states[1:]):
    graph.add(gu.WnoaMotionFactorAugmentedPoint3(left, right, q_psd))

for index, (t, ranges) in enumerate(zip(measurement_times, measured_ranges)):
    if index == 0:
        position_key = states[0].pose
    else:
        interpolated_state = gtsam.StateData(
            gtsam.symbol("p", 100 + index),
            gtsam.symbol("v", 100 + index),
            float(t),
        )
        position_key = interpolated_state.pose

    for landmark, distance in zip(landmarks, ranges):
        noise = gtsam.noiseModel.Isotropic.Sigma(1, 2 * distance * range_sigma)
        range_factor = gu.KnownLandmarkSquaredRangeFactorAugmentedPoint3(
            position_key, landmark, float(distance), noise
        )
        if index == 0:
            graph.add(range_factor)
        else:
            graph.add(gu.WnoaInterpFactorAugmentedPoint3(
                range_factor, set(states), {interpolated_state}, q_psd, True, True
            ))

print(f"Knot states: {len(states)}")
print(f"Range measurements: {measured_ranges.size}")
print(f"Factor graph: {graph.size()} factors, {initial.size()} variables")

Knot states: 4
Range measurements: 28
Factor graph: 31 factors, 8 variables


## Estimate the knot states

Levenberg–Marquardt minimizes the sum of the WNOA motion costs and squared-range measurement costs. Because the problem is nonlinear, the result depends on the initial estimate; the perturbed knot states above provide a nearby starting trajectory.

In [6]:
result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()


def knot_values(values):
    positions = np.array([
        np.asarray(gu.atAugmentedPoint3(state.pose, values).vector()).reshape(3)
        for state in states
    ])
    velocities = np.array([
        np.asarray(values.atVector(state.velocity)).reshape(3)
        for state in states
    ])
    return positions, velocities


initial_positions, initial_velocities = knot_values(initial)
estimated_positions, estimated_velocities = knot_values(result)

print(f"Initial graph error: {graph.error(initial):.4f}")
print(f"Final graph error:   {graph.error(result):.4f}")
print(f"Knot position RMSE: {np.sqrt(np.mean(np.sum((estimated_positions - true_position(knot_times))**2, axis=1))):.4f} m")

Initial graph error: 868.9009
Final graph error:   3.1728
Knot position RMSE: 0.0495 m


## Trajectory estimate

The estimated knot positions and velocities define a continuous trajectory between the knots. For the Euclidean WNOA model, the conditional mean follows cubic Hermite interpolation. The 3D plot compares that trajectory with ground truth and the initial estimate; the landmarks are shown for spatial reference.

In [10]:
def interpolate_positions(times, positions, velocities):
    segment = np.clip(
        np.searchsorted(knot_times, times, side="right") - 1,
        0, len(knot_times) - 2,
    )
    dt = knot_times[segment + 1] - knot_times[segment]
    u = (times - knot_times[segment]) / dt
    return (
        (2 * u**3 - 3 * u**2 + 1)[:, None] * positions[segment]
        + (u**3 - 2 * u**2 + u)[:, None] * dt[:, None] * velocities[segment]
        + (-2 * u**3 + 3 * u**2)[:, None] * positions[segment + 1]
        + (u**3 - u**2)[:, None] * dt[:, None] * velocities[segment + 1]
    )


plot_times = np.linspace(knot_times[0], knot_times[-1], 200)
true_path = true_position(plot_times)
initial_path = interpolate_positions(plot_times, initial_positions, initial_velocities)
estimated_path = interpolate_positions(plot_times, estimated_positions, estimated_velocities)

fig = go.Figure()
fig.add_scatter3d(x=true_path[:, 0], y=true_path[:, 1], z=true_path[:, 2], mode="lines", name="Ground truth", line=dict(width=5))
fig.add_scatter3d(x=initial_path[:, 0], y=initial_path[:, 1], z=initial_path[:, 2], mode="lines", name="Initial estimate", line=dict(width=3, dash="dash"))
fig.add_scatter3d(x=estimated_path[:, 0], y=estimated_path[:, 1], z=estimated_path[:, 2], mode="lines", name="WNOA estimate", line=dict(width=5))
fig.add_scatter3d(x=estimated_positions[:, 0], y=estimated_positions[:, 1], z=estimated_positions[:, 2], mode="markers", name="Estimated knots", marker=dict(size=5))
fig.add_scatter3d(x=landmarks[:, 0], y=landmarks[:, 1], z=landmarks[:, 2], mode="markers", name="Known landmarks", marker=dict(size=6, symbol="diamond"))
fig.update_layout(title="Range-only trajectory estimate", scene=dict(xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)", aspectmode="data"), margin=dict(l=0, r=0, b=0, t=45))
fig.show()

## Position error

The Euclidean distance between the estimated and true positions measures trajectory accuracy in meters. This plot evaluates the continuous trajectories, not just their knot positions.

In [12]:
initial_error = np.linalg.norm(initial_path - true_path, axis=1)
estimated_error = np.linalg.norm(estimated_path - true_path, axis=1)

fig = go.Figure()
fig.add_scatter(x=plot_times, y=initial_error, mode="lines", name="Initial estimate", line=dict(dash="dash"))
fig.add_scatter(x=plot_times, y=estimated_error, mode="lines", name="WNOA estimate")
fig.update_layout(title="Trajectory position error", xaxis_title="Time (s)", yaxis_title="Position error (m)")
fig.show()

print(f"Mean position error, initial: {initial_error.mean():.4f} m")
print(f"Mean position error, WNOA:    {estimated_error.mean():.4f} m")

Mean position error, initial: 0.2222 m
Mean position error, WNOA:    0.0304 m


The range observations constrain the trajectory at their actual measurement times, even though only the knot states are estimated. The WNOA prior couples neighboring positions and velocities, giving a smooth trajectory between observations. Unlike single-point range localization, this graph estimates **a continuous path** from asynchronous range measurements.